# Opportune sabotage — make Neural A* ignore a narrow shortcut without closing it

**Scenario.** The shortest route passes through a **gate**: a 3-cell opening in a wall. Every
route that avoids it is much longer (the *detour*). The attacker flips a few cells near the
gate, under two constraints:

1. the gate is **never fully closed** — at least `min_open` of its cells stay free;
2. the gate **stays the right choice** — on the altered map, the optimum through the gate is
   strictly shorter than the optimum with the gate closed.

Because of (2), classical A\* on the altered map still takes the gate, so if Neural A\* goes
round it is wrong by the project's error rule (deviation from the *altered* map's optimum).
**Success** = Neural A\*'s route no longer touches the gate.

**Why the maps are built, not taken from the benchmark.** The benchmark's openings are as wide
as a room (~10 cells); section 1 checks that no test map has a narrow passage on its optimal
route. `src/sabotage.py` draws maze-like maps — thin walls, 32 × 32, same encoding — with one
3-cell gate and a long detour, in random orientations, and keeps those the *clean* planner
solves optimally through the gate (a sabotage only means something where the planner was right).

**The attack.** Greedy, like step 9: each round tries every cell within the encoder's reach
(Chebyshev radius 5) of the gate, drops additions that break the constraints, runs the planner
on the rest in one batch, and keeps the one that lengthens the route most (gate avoidance
first). The control draws the same number of random admissible flips from the same cells.

Run from the repository root with the project venv:
`.venv/bin/python -m jupyterlab notebooks/03_sabotage.ipynb`

In [ ]:
import sys
sys.path.insert(0, "..")

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

from scripts.demo import explain as route_reading
from scripts.demo import order_path
from src import config
from src.data import load_split
from src.exact_solver import solve_dijkstra
from src.planner import load_pretrained_neural_astar
from src.sabotage import (assess, find_gates, random_sabotage, sabotage_attack, scenario_candidates,
                          select_scenarios)
from src.swap import run_swap

SEED = 0
config.set_global_seeds(SEED)
planner = load_pretrained_neural_astar()
instances = load_split("test").instances(seed=SEED)

In [ ]:
WALLS = ListedColormap(["#1b2530", "#f8fafb"])


def draw(ax, design, start, goal, route=None, optimal=None, expanded=None, field=None,
         cmap="viridis", vmin=None, vmax=None, gate=(), flips=(), title=""):
    # Map (or a per-cell field over the free cells), gate (purple), flips (yellow), routes.
    if field is None:
        ax.imshow(design, cmap=WALLS, vmin=0, vmax=1)
    else:
        ax.imshow(np.zeros_like(design), cmap=ListedColormap(["#1b2530"]))
        ax.imshow(np.ma.masked_where(design == 0, field), cmap=cmap, vmin=vmin, vmax=vmax)
    if expanded is not None:
        ax.imshow(np.ma.masked_where(expanded == 0, expanded), cmap=ListedColormap(["#6ea0de"]), alpha=0.45)
    if optimal is not None:
        rc = np.array(optimal)
        ax.plot(rc[:, 1], rc[:, 0], color="#149e8e", lw=5, alpha=0.5)
    if route is not None:
        rc = np.array(order_path(route, start, goal))
        ax.plot(rc[:, 1], rc[:, 0], color="#f05a28", lw=2)
    for r, c in gate:
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec="#a46cff", lw=2.5))
    for r, c in flips:
        ax.add_patch(plt.Rectangle((c - 0.42, r - 0.42), 0.84, 0.84, fill=False, ec="#ffc21a", lw=2))
    ax.scatter([start[1]], [start[0]], c="#257a4d", s=50, zorder=5)
    ax.scatter([goal[1]], [goal[0]], c="#c9342a", s=50, zorder=5)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

## 1. Does the benchmark have natural gates?

For every cell of one optimal route, the free run through it along the row and the column; runs
of ≤ 4 cells whose closure lengthens the optimum by at least one move count as gates.

In [ ]:
natural = [len(find_gates(i.map_design, i.start_rc, i.goal_rc, max_width=4)) for i in instances]
print(f"test maps with a natural gate (<= 4 cells): {sum(n > 0 for n in natural)} / {len(natural)}")

## 2. The scenarios

How often the clean planner already gets these maps right (optimal and through the gate) is
itself a result: they are maze-like but not benchmark maps.

In [ ]:
pool = scenario_candidates(range(60))
clean = [assess(planner, sc) for sc in pool]
pool_table = pd.DataFrame([{
    "scenario": sc.name, "layout": sc.layout, "optimum": sc.base_cost, "detour": sc.closed_cost,
    "planner moves": a["moves"], "planner optimal": a["moves"] <= a["optimal_cost"] + 1e-6,
    "planner uses gate": a["planner_uses_gate"],
} for sc, a in zip(pool, clean)])
print(f"{len(pool)} generated scenarios: clean planner optimal on "
      f"{pool_table['planner optimal'].mean():.0%}, through the gate on {pool_table['planner uses gate'].mean():.0%}")
pool_table.groupby("layout")[["planner optimal", "planner uses gate"]].mean().round(2)

In [ ]:
scenarios = select_scenarios(planner, n=3, seeds=range(400))
assert scenarios, "no scenario where the clean planner takes the gate optimally; widen the seeds"
pd.DataFrame([{"scenario": sc.name, "layout": sc.layout, "gate": sc.gate, "optimum": sc.base_cost,
               "detour": sc.closed_cost, "detour ratio": round(sc.detour_ratio, 2)} for sc in scenarios])

In [ ]:
fig, axes = plt.subplots(2, len(scenarios), figsize=(5 * len(scenarios), 10), squeeze=False)
for j, sc in enumerate(scenarios):
    inst, a = sc.instance, assess(planner, sc)
    optimal = solve_dijkstra(inst.map_design, inst.start_rc, inst.goal_rc).path
    draw(axes[0, j], inst.map_design, inst.start_rc, inst.goal_rc, route=a["path_map"], optimal=optimal,
         expanded=a["histories"], gate=sc.gate,
         title=f"{sc.name}: {a['moves']:.0f} moves (opt {sc.base_cost:.0f}, detour {sc.closed_cost:.0f})")
    draw(axes[1, j], inst.map_design, inst.start_rc, inst.goal_rc, route=a["path_map"], field=a["cost_map"],
         cmap="Blues", vmin=0, vmax=1, gate=sc.gate, title="cost map (clean)")
fig.suptitle("purple = gate, orange = Neural A*, green = shortest route, blue = expanded", y=0.995)
fig.tight_layout()
plt.show()

## 3. The attack

Budget 4, radius 5, at least one gate cell left open. The search is run to the full budget so
the success curve over k is available; the flips reported for each scenario are those of the
first round that fools the planner.

In [ ]:
BUDGET, RADIUS, MIN_OPEN = 4, 5, 1
attacks = {sc.name: sabotage_attack(planner, sc, budget=BUDGET, radius=RADIUS, min_open=MIN_OPEN,
                                    stop_on_success=False) for sc in scenarios}

rounds = pd.DataFrame([{"scenario": name, **{k: v for k, v in r.items() if k != "flips"},
                        "flips": r["flips"]} for name, res in attacks.items() for r in res.rounds])
first_success = {name: next((r["n_flips"] for r in res.rounds if r["success"]), None)
                 for name, res in attacks.items()}
print("first k that fools the planner:", first_success)
rounds[["scenario", "n_flips", "moves", "optimal_cost", "closed_cost", "opt_ratio", "planner_uses_gate",
        "shortcut_optimal", "gate_open", "success", "expanded", "classical_expanded", "flips"]]

In [ ]:
def final_flips(name):
    res = attacks[name]
    k = first_success[name]
    row = next(r for r in res.rounds if r["n_flips"] == k) if k is not None else res.rounds[-1]
    return row["flips"]


fig, axes = plt.subplots(len(scenarios), 4, figsize=(19, 4.9 * len(scenarios)), squeeze=False)
for i, sc in enumerate(scenarios):
    inst = sc.instance
    before, after = assess(planner, sc), assess(planner, sc, final_flips(sc.name))
    opt_after = solve_dijkstra(after["design"], inst.start_rc, inst.goal_rc).path
    delta = after["cost_map"] - before["cost_map"]
    lim = max(np.abs(delta).max(), 1e-6)
    draw(axes[i, 0], inst.map_design, inst.start_rc, inst.goal_rc, route=before["path_map"],
         optimal=solve_dijkstra(inst.map_design, inst.start_rc, inst.goal_rc).path, gate=sc.gate,
         title=f"{sc.name} clean: {before['moves']:.0f} moves")
    draw(axes[i, 1], after["design"], inst.start_rc, inst.goal_rc, route=after["path_map"], optimal=opt_after,
         expanded=after["histories"], gate=sc.gate, flips=after["flips"],
         title=f"{len(after['flips'])} flips: {after['moves']:.0f} moves (opt {after['optimal_cost']:.0f})"
               + (" — FOOLED" if after["success"] else ""))
    draw(axes[i, 2], after["design"], inst.start_rc, inst.goal_rc, field=delta, cmap="RdBu_r", vmin=-lim, vmax=lim,
         gate=sc.gate, flips=after["flips"], title="cost change (red = costlier)")
    draw(axes[i, 3], after["design"], inst.start_rc, inst.goal_rc, route=after["path_map"], field=after["cost_map"],
         cmap="Blues", vmin=0, vmax=1, gate=sc.gate, flips=after["flips"], title="cost map after")
fig.suptitle("yellow = flipped cells, purple = gate", y=0.998)
fig.tight_layout()
plt.show()

## 4. Which route did the damage travel? (the 2 × 2 swap of step 7)

The search is rerun with only the new costs (`C1O0`) and with only the new walls (`C0O1`). If
the new costs alone send the planner round, the encoder was fooled; if the old costs fail on
the new walls, the search was.

In [ ]:
swap_rows = []
for sc in scenarios:
    flips = final_flips(sc.name)
    if not flips:
        continue
    sw = run_swap(planner, sc.instance, flips)
    swap_rows.append({"scenario": sc.name, "flips": len(flips),
                      **{f"moves {k}": v for k, v in sw.path_cost.items()},
                      "optimum (old walls)": sw.references["O0"][0], "optimum (new walls)": sw.references["O1"][0],
                      "reading": route_reading(sw.path_cost, 0.5)})
pd.DataFrame(swap_rows)

## 5. Targeted vs random, and how much the constraint matters

* **Random control**: `k` random admissible flips from the same neighbourhood, 200 draws per
  scenario and budget.
* **Constraint strength**: the greedy attack again with `min_open` = 1, 2 and 3. With 3, no
  gate cell may be touched: the planner must be lured away by changes *around* the gate only.

In [ ]:
control = []
for sc in scenarios:
    res = attacks[sc.name]
    for k in range(1, BUDGET + 1):
        greedy = next((r for r in res.rounds if r["n_flips"] == k), None)
        rnd = random_sabotage(planner, sc, budget=k, draws=200, radius=RADIUS, min_open=MIN_OPEN, seed=SEED)
        control.append({"scenario": sc.name, "k": k,
                        "greedy fooled": None if greedy is None else greedy["success"],
                        "greedy opt ratio": None if greedy is None else greedy["opt_ratio"],
                        "random success rate": rnd["success_rate"], "random mean opt ratio": rnd["mean_opt_ratio"],
                        "random draws": rnd["draws"]})
control = pd.DataFrame(control)
control.round(3)

In [ ]:
strength = []
for sc in scenarios:
    for min_open in (1, 2, 3):
        res = sabotage_attack(planner, sc, budget=BUDGET, radius=RADIUS, min_open=min_open)
        strength.append({"scenario": sc.name, "min_open": min_open, "fooled": res.success,
                         "flips used": len(res.flips), "final moves": res.rounds[-1]["moves"],
                         "optimum": res.rounds[-1]["optimal_cost"], "gate cells left open": res.rounds[-1]["gate_open"],
                         "maps evaluated": res.evaluated})
strength = pd.DataFrame(strength)
strength

## 6. Save

In [ ]:
out = {
    "natural_gate_maps": int(sum(n > 0 for n in natural)),
    "generated_pool": {"n": len(pool), "clean_optimal": float(pool_table["planner optimal"].mean()),
                       "clean_uses_gate": float(pool_table["planner uses gate"].mean())},
    "settings": {"budget": BUDGET, "radius": RADIUS, "min_open": MIN_OPEN},
    "scenarios": [{"name": sc.name, "layout": sc.layout, "seed": sc.seed, "gate": sc.gate,
                   "base_cost": sc.base_cost, "closed_cost": sc.closed_cost,
                   "first_success_k": first_success[sc.name], "flips": final_flips(sc.name)} for sc in scenarios],
    "rounds": rounds.to_dict(orient="records"),
    "swap": swap_rows,
    "random_control": control.to_dict(orient="records"),
    "constraint_strength": strength.to_dict(orient="records"),
}
path = config.RESULTS_DIR / "sabotage.json"
path.write_text(json.dumps(out, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("wrote", path)

## How to read the outcome

* **Fooled with few flips, random rarely** → the shortcut is a cheap target: a handful of cells
  placed by the planner's own response beat chance by a wide margin, while the true optimum
  never stopped going through the gate.
* **The swap reading** says whether the planner was steered by its **costs** (the encoder
  painted the gate expensive) or by the **search** (the old costs fail once the opening narrows).
  Step 7 found route damage encoder-led on the benchmark; this tests it on a targeted case.
* **`min_open` = 3 still fooled** is the strongest form: the gate is untouched and the planner
  still goes round, purely from what changed in its 11 × 11 view of the gate.

The demo shows the same thing live: *Shortcut maps* loads these scenarios, *Sabotage the
shortcut* adds one constrained flip per click.